# Data sources 5/6: all gists of a user

`gist_catalog(username)` describes every gist - files, stars, comments, forks, commits and
size - and `clone_gists(gists, path)` clones them as they are, one folder per gist id.
GraphQL does the catalog (100 gists per request; it needs a token), REST counts commits.

In [ ]:
from gitrecon.config import load_env

load_env()  # GH_TOKEN for GraphQL

In [ ]:
from gitrecon.sources.gists import gist_catalog

catalog = gist_catalog("sarverott")            # privacy="all" adds secret gists - with the owner's token
len(catalog), catalog[0]

In [ ]:
size = sum(g["size"] for g in catalog)
print(f"{len(catalog)} gists, {size / 2**20:.1f} MiB of files, "
      f"{sum(g['commits'] for g in catalog)} commits, {sum(g['stars'] for g in catalog)} stars")
for g in sorted(catalog, key=lambda g: -g["commits"])[:5]:
    print(f"{g['commits']:>4} commits  {g['gistID']}  {', '.join(g['filelist'])[:60]}")

## Cloning

The gist `https://gist.github.com/<gistID>.git` goes to `<path>/<gistID>`; the path is created
when missing, existing clones are kept (or fast-forwarded with `update=True`). Here only two,
into a temporary folder - the real target is below, switched off.

In [ ]:
import tempfile

from gitrecon.sources.gists import clone_gists

results = clone_gists(catalog[:2], tempfile.mkdtemp(prefix="gists-"))
results

In [ ]:
CLONE_ALL = False  # every gist, as it is
if CLONE_ALL:
    clone_gists(catalog, "~/__WORKSHOP/forge/sarverott/my-gists", progress=print)

CLI: `gitrecon gist-catalog sarverott [--json|--urls]`,
`gitrecon gist-clone sarverott ~/__WORKSHOP/forge/sarverott/my-gists --dry-run` (size first),
then without `--dry-run`.